%% [markdown]
# Nucleotide Transformer inference parity — CPU / CUDA / Trainium  (DEV mode)
Runs the vendored **full model** (`EsmForMaskedLM`, encoder + MLM head) on every available device and
checks the outputs (logits + last hidden state) match. CPU is the reference; `cuda` auto-skips when
absent; `neuron` runs on the Trainium core. Pin a free core:
`NEURON_RT_VISIBLE_CORES=0 jupyter nbconvert --to notebook --execute 01_inference_parity.ipynb`.

In [1]:
# %%
import os
os.environ.setdefault("NEURON_RT_VISIBLE_CORES", "0")
import sys
sys.path.insert(0, os.path.abspath("../src"))
import torch
import torch.nn.functional as F
import nucleotide_transformer_reference as R

[W709 03:21:27.441137938 OperatorEntry.cpp:208] Warning: Warning only once for all operators,  other operators may also be overridden.
  Overriding a previously registered kernel for the same operator and the same dispatch key
  operator: aten::gather.out(Tensor self, int dim, Tensor index, *, bool sparse_grad=False, Tensor(a!) out) -> Tensor(a!)
    registered at /pytorch/build/aten/src/ATen/RegisterSchema.cpp:6
  dispatch key: PrivateUse1
  previous kernel: registered at /pytorch/build/aten/src/ATen/RegisterCPU_3.cpp:7637
       new kernel: registered at NeuronDispatcher.cpp:0 (function operator())
/home/user/miniconda3/envs/torch-neuron/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def devices():
    devs = ["cpu"]
    if torch.cuda.is_available():
        devs.append("cuda")
    try:
        import torch_neuronx  # noqa: F401
        devs.append("neuron")
    except Exception as e:
        print("neuron unavailable:", e)
    return devs

In [3]:
DEVICES = devices()
print("torch", torch.__version__, "| devices:", DEVICES, "| model:", R.MODEL_ID)

torch 2.11.0+cpu | devices: ['cpu', 'neuron'] | model: InstaDeepAI/nucleotide-transformer-v2-50m-multi-species


In [4]:
# %% [markdown]
# ## Run the full model on each device
# %%
def run(device):
    model = R.load(device)
    ids, mask = R.build_inputs()
    with torch.no_grad():
        out = model(ids.to(device), mask.to(device))
    if device == "neuron":
        import torch_neuronx; torch_neuronx.synchronize()
    return tuple(t.detach().float().cpu() for t in out)

In [5]:
results = {d: run(d) for d in DEVICES}
for name, t in zip(R.OUTPUT_ORDER, results["cpu"]):
    print(f"cpu {name:8s} shape={tuple(t.shape)}")

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


[transformers] Detected the usage of `get_extended_attention_mask`: This function is deprecated and will be removed in v5.12.0. Please use the new API in `transformers.masking_utils`


cpu logits   shape=(2, 64, 4107)
cpu hidden   shape=(2, 64, 512)


In [6]:
# %% [markdown]
# ## Check every device matches CPU (per-output cosine + max-abs)
# %%
def cos(a, b): return F.cosine_similarity(a.flatten(), b.flatten(), dim=0).item()

In [7]:
ref, all_ok = results["cpu"], True
for d in DEVICES:
    if d == "cpu":
        continue
    print(f"\n{d} vs cpu:")
    for name, a, b in zip(R.OUTPUT_ORDER, ref, results[d]):
        c = cos(a, b); mab = (a - b).abs().max().item(); ok = c >= 0.99
        all_ok = all_ok and ok
        print(f"  {name:8s} cosine={c:.6f}  max-abs={mab:.3e}  {'OK' if ok else 'FAIL'}")


neuron vs cpu:
  logits   cosine=1.000011  max-abs=1.316e-04  OK
  hidden   cosine=1.000001  max-abs=1.484e-05  OK


In [8]:
print("\nINFERENCE PARITY:", "PASS" if all_ok else "FAIL")
assert all_ok, "full-model outputs diverged across devices"


INFERENCE PARITY: PASS
